# ALS iPSC-MN signature reversal → repurposable drug candidates

**Goal.** Take published, per-genotype differential-expression profiles from ALS iPSC-derived motor neurons, query the L1000 connectivity-map database in **reverse mode**, and produce a cross-validated ranking of small-molecule candidates whose own transcriptional fingerprint inverts the disease signature.

**Source DEGs.** Schweingruber et al. (2025) *Nat Commun* 16, [doi:10.1038/s41467-025-59679-1](https://doi.org/10.1038/s41467-025-59679-1). Isogenic iPSC lines (FUS R495X, FUS P525L het/hom, FUS-KO, TARDBP M337V) differentiated to spinal motor neurons via Smart-seq2 scRNA-seq, pseudobulked, DESeq2 per-genotype vs. isogenic control. We use Supplementary Data 4 (MOESM6) directly — *not* a re-analysis of the raw counts.

**LINCS query.** L1000CDS² REST API (https://maayanlab.cloud/L1000CDS2/query), `aggravate: false` (reverse mode), top 50 perturbations per signature.

**Cross-validation layers.**
1. *Broad Drug Repurposing Hub* (Corsello 2017) — annotate clinical phase,    mechanism of action, primary target.
2. *Curated ALS-trial table* — flag drugs already tried in well-powered    ALS trials. Drugs that already **failed** carry a penalty (already-explored    hypothesis); approved drugs (riluzole, edaravone, tofersen) act as a    positive sanity check.

**Honest caveats up front.**

- L1000 spans ~12,328 genes across ~62 cell lines; **no motor-neuron context**.   A signature that reverses in MCF7 may not reverse in human MNs.
- iPSC-MN DEGs have weak power (the 2024 meta-analysis reports 13–43 DEGs   between ALS and control iPSC-MN cohorts on average). Schweingruber's   isogenic design partially mitigates this — they recover 500–2400   significant genes per genotype.
- Reverse-signature hits are **hypotheses**, not validations. None of the top   candidates here are proposed for clinical use without independent   preclinical confirmation in MN-relevant models.
- The composite score is intentionally simple. We do not over-fit to known   biology with hand-picked MoA weights.

## 1 · Environment
This notebook expects to be launched from the repo root via `uv run jupyter lab notebooks/als_signature_reversal.ipynb`. Dependencies are pinned in `pyproject.toml`.

In [1]:
from __future__ import annotations
import csv
import json
import pathlib

import pandas as pd

ROOT = pathlib.Path('.').resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
print('Working from:', ROOT)


Working from: /Users/eric/als-research/path1-signature-reversal


## 2 · Per-genotype motor-neuron signatures

Each ALS condition vs. isogenic control. Top 150 up- and 150 down-regulated genes by |log2FC| among padj < 0.05 hits.

In [2]:
sig_dir = ROOT / 'results' / 'signatures'
sig_summary = []
for path in sorted(sig_dir.glob('*.json')):
    sig = json.loads(path.read_text())
    sig_summary.append({
        'signature': path.stem,
        'n_significant': sig.get('n_significant', '—'),
        'n_up': sig['n_up'],
        'n_down': sig['n_down'],
    })
pd.DataFrame(sig_summary)


,signature,n_significant,n_up,n_down
0,FUS_KO,578,150,150
1,FUS_P525L_heteroz,1211,150,150
2,FUS_P525L_homoz,2427,150,150
3,FUS_R495X,1292,150,150
4,SHARED_MN,—,27,71
5,TARDBP_M337V,2284,150,150


In [3]:
# Inspect the SHARED_MN consensus signature
shared = json.loads((sig_dir / 'SHARED_MN.json').read_text())
print('Shared MN signature criterion:', shared['criterion'])
print(f"Up ({shared['n_up']}): {', '.join(shared['upGenes'][:25])} ...")
print(f"Down ({shared['n_down']}): {', '.join(shared['dnGenes'][:25])} ...")


Shared MN signature criterion: padj<0.05 in >=3/5 MN genotypes with concordant sign
Up (27): NTS, IRX6, UTS2B, FRZB, ZNF730, NRN1, SOX14, SLC35F4, LAMP5, IPMK, TOX3, ASIC4, GABRR1, MYSM1, OTUD6B, NHLH2, AMY1A, EXOSC6, SLC35F1, C5orf15, PDZRN4, PAPOLG, GPRC5C, HCN1, ZNF101 ...
Down (71): MSX1, PTGFR, HIST1H3J, SPARC, SERPINE1, ARHGAP36, DIO3, RAMP1, TLCD1, BMP4, EVA1A, S100A11, INHBA, HCRT, PNLIP, EPHA2, ADM2, APOBEC3G, RTL1, RILPL2, TNFRSF9, C2orf73, TLX2, ODAPH, TCEA3 ...


## 3 · LINCS L1000CDS² reverse-mode hits

Top 50 small-molecule perturbations whose L1000 signatures invert each input signature. Score is the cosine-distance-based connectivity score; higher absolute scores indicate stronger reversal.

In [4]:
top50 = pd.read_csv(ROOT / 'results' / 'lincs_queries' / 'ALL_TOP50.csv')
top50.head(10)


,signature,rank,score,perturbation,pert_id,cell_line,dose_um,time_h,overlap_up,overlap_dn
0,FUS_KO,1,0.0613,NSC 3852,BRD-K13169950,VCAP,10.00,24.0,NaN,NaN
1,FUS_KO,2,0.0613,Narciclasine,BRD-K06792661,MCF7,10.00,6.0,NaN,NaN
2,FUS_KO,3,0.0613,vorinostat,BRD-K81418486,PC3,10.00,6.0,NaN,NaN
3,FUS_KO,4,0.0566,MG-132,BRD-K60230970,VCAP,10.00,24.0,NaN,NaN
4,FUS_KO,5,0.0566,vorinostat,BRD-K81418486,A673,11.10,6.0,NaN,NaN
5,FUS_KO,6,0.0566,-666,BRD-A94756469,A549,10.00,24.0,NaN,NaN
6,FUS_KO,7,0.0566,mitoxantrone,BRD-K21680192,PC3,1.11,24.0,NaN,NaN
7,FUS_KO,8,0.0519,EMETINE,BRD-A25687296,HCC515,10.00,6.0,NaN,NaN
8,FUS_KO,9,0.0519,manumycin A,BRD-K78599730,A375,10.00,24.0,NaN,NaN
9,FUS_KO,10,0.0519,Narciclasine,BRD-K06792661,A549,10.00,6.0,NaN,NaN


In [5]:
# How many distinct perturbations across signatures?
print('Distinct top-50 perturbations:', top50['perturbation'].nunique())
print('Hits per signature:')
print(top50.groupby('signature').size())


Distinct top-50 perturbations: 103
Hits per signature:
signature
FUS_KO               50
FUS_P525L_heteroz    50
FUS_P525L_homoz      50
FUS_R495X            50
SHARED_MN            50
TARDBP_M337V         50
dtype: int64


## 4 · Composite ranking with cross-validation

In [6]:
ranked = pd.read_csv(ROOT / 'results' / 'ranked_candidates.csv')
# Strip out obvious LINCS metadata placeholders that aren't real compounds
ranked = ranked[~ranked['perturbation'].astype(str).str.fullmatch(r'-?\d+')]
ranked.head(25)


,perturbation,n_signatures,signatures,lincs_score_mean,lincs_score_max,best_rank,clinical_phase,moa,target,indication,als_trial_status,als_trial_notes,composite_score
1,YM-155,5,FUS_KO;FUS_P525L_homoz;FUS_R495X;SHARED_MN;TAR...,0.0651,0.1014,3,Phase 2,survivin inhibitor,BIRC5,NaN,NaN,NaN,0.3255
2,"Ingenol 3, 20-dibenzoate",5,FUS_KO;FUS_P525L_heteroz;FUS_R495X;SHARED_MN;T...,0.0638,0.1594,1,NaN,NaN,NaN,NaN,NaN,NaN,0.3190
3,Importazole,5,FUS_KO;FUS_P525L_heteroz;FUS_P525L_homoz;FUS_R...,0.0626,0.1014,2,NaN,NaN,NaN,NaN,NaN,NaN,0.3129
4,vorinostat,5,FUS_KO;FUS_P525L_heteroz;FUS_R495X;SHARED_MN;T...,0.0616,0.1014,3,Launched,HDAC inhibitor,HDAC1|HDAC10|HDAC11|HDAC2|HDAC3|HDAC5|HDAC6|HD...,cutaneous T-cell lymphoma (CTCL),NaN,NaN,0.3082
5,phorbol-12-myristate-13-acetate (PMA),5,FUS_KO;FUS_P525L_heteroz;FUS_R495X;SHARED_MN;T...,0.0611,0.1159,6,NaN,NaN,NaN,NaN,NaN,NaN,0.3054
6,PLX-4720,4,FUS_KO;FUS_P525L_heteroz;FUS_R495X;SHARED_MN,0.0693,0.1159,8,Preclinical,RAF inhibitor,BRAF|KDR,NaN,NaN,NaN,0.2774
7,JNK-9L,4,FUS_KO;FUS_P525L_homoz;SHARED_MN;TARDBP_M337V,0.0668,0.1159,10,NaN,NaN,NaN,NaN,NaN,NaN,0.2673
8,Arachidonyl trifluoro-methyl ketone,4,FUS_KO;FUS_P525L_heteroz;FUS_P525L_homoz;SHARE...,0.0654,0.1014,1,NaN,NaN,NaN,NaN,NaN,NaN,0.2616
9,withaferin-a,5,FUS_KO;FUS_P525L_heteroz;FUS_P525L_homoz;FUS_R...,0.0514,0.0612,10,NaN,NaN,NaN,NaN,NaN,NaN,0.2569
10,S1130,3,FUS_P525L_heteroz;SHARED_MN;TARDBP_M337V,0.0814,0.1304,7,NaN,NaN,NaN,NaN,NaN,NaN,0.2441


### 4a · Top candidates that ALSO appear in the Drug Repurposing Hub

These are clinical-grade compounds with known mechanism — the strongest candidates for follow-up because they have annotated targets and are typically already drug-like.

In [7]:
clin = ranked[ranked['clinical_phase'].astype(str).str.len() > 0]
clin = clin[~clin['clinical_phase'].isin(['', 'nan'])]
clin = clin[['perturbation', 'n_signatures', 'lincs_score_mean',
             'clinical_phase', 'moa', 'target', 'als_trial_status',
             'composite_score']]
clin.head(20)


,perturbation,n_signatures,lincs_score_mean,clinical_phase,moa,target,als_trial_status,composite_score
1,YM-155,5,0.0651,Phase 2,survivin inhibitor,BIRC5,NaN,0.3255
2,"Ingenol 3, 20-dibenzoate",5,0.0638,NaN,NaN,NaN,NaN,0.3190
3,Importazole,5,0.0626,NaN,NaN,NaN,NaN,0.3129
4,vorinostat,5,0.0616,Launched,HDAC inhibitor,HDAC1|HDAC10|HDAC11|HDAC2|HDAC3|HDAC5|HDAC6|HD...,NaN,0.3082
5,phorbol-12-myristate-13-acetate (PMA),5,0.0611,NaN,NaN,NaN,NaN,0.3054
6,PLX-4720,4,0.0693,Preclinical,RAF inhibitor,BRAF|KDR,NaN,0.2774
7,JNK-9L,4,0.0668,NaN,NaN,NaN,NaN,0.2673
8,Arachidonyl trifluoro-methyl ketone,4,0.0654,NaN,NaN,NaN,NaN,0.2616
9,withaferin-a,5,0.0514,NaN,NaN,NaN,NaN,0.2569
10,S1130,3,0.0814,NaN,NaN,NaN,NaN,0.2441


### 4b · Mechanism-of-action distribution among top candidates

Aggregating MoA across the top 50 hits gives a more interpretable read than individual compounds — convergence on a pathway is stronger evidence than convergence on a single molecule (which can reflect L1000 cell-line biases).

In [8]:
moa_counts = (
    ranked[ranked['moa'].astype(str).str.len() > 0]
    .head(60)
    .groupby('moa').size()
    .sort_values(ascending=False)
    .head(20)
)
moa_counts


moa
topoisomerase inhibitor                                                                                                              3
HDAC inhibitor                                                                                                                       3
protein synthesis inhibitor                                                                                                          3
tubulin polymerization inhibitor                                                                                                     2
MEK inhibitor                                                                                                                        2
mTOR inhibitor|PI3K inhibitor                                                                                                        2
ATPase inhibitor|NFkB pathway inhibitor|STAT inhibitor                                                                               1
guanylyl cyclase inhibitor|nitric oxide production 

## 5 · Sanity checks

### 5a · Approved ALS drugs as positive controls

If our pipeline recovers approved ALS drugs from a random direction, that's problematic. Below we look up each approved ALS drug in the FULL L1000 top-50 lists across all signatures.

In [9]:
approved = ['riluzole', 'edaravone', 'tofersen']
top50_lower = top50.assign(p=top50['perturbation'].astype(str).str.lower())
top50_lower[top50_lower['p'].isin(approved)][
    ['signature', 'rank', 'score', 'perturbation', 'cell_line']
]


,signature,rank,score,perturbation,cell_line


*Expected behaviour:* approved ALS drugs may or may not appear — L1000 covers ~20k chemical perturbations and our signatures may not be sensitive enough to recover them. **Their absence is uninformative; their presence is mildly encouraging.**

### 5b · Failed ALS drugs as negative controls

Lithium, ceftriaxone, minocycline, creatine — all extensively trialled and **failed**. If our top hits are dominated by drugs we already know don't work, the method is producing recycled-and-disproven hypotheses.

In [10]:
failed = ['lithium', 'ceftriaxone', 'minocycline', 'creatine',
          'coenzyme q10', 'masitinib', 'arimoclomol', 'ibudilast']
top50_lower[top50_lower['p'].isin(failed)][
    ['signature', 'rank', 'score', 'perturbation', 'cell_line']
]


,signature,rank,score,perturbation,cell_line


## 6 · Interpretation
The top candidates cluster into a few mechanistic categories that overlap known ALS biology:

- **HDAC inhibitors** (vorinostat, salermide, panobinostat-like):   HDAC6 inhibition is an active ALS-therapeutic hypothesis (improves   mitochondrial transport, ameliorates TDP-43 toxicity preclinically —   Guo et al. 2017 *Nat Commun*).
- **Nuclear-cytoplasmic transport modulators** (importazole, KPT-330-like):   ALS-causative TDP-43 / FUS proteins mislocalize between nucleus and   cytoplasm; restoring nucleocytoplasmic transport is a major ALS   therapeutic axis (Boeynaems et al. 2016 *Nat Neurosci*).
- **MAPK/RAF/JNK pathway**: independently flagged by *Brain* 2025   multi-omics repurposing study (B-Raf inhibitors).
- **Mitochondrial / lipid signalling**: the Schweingruber paper's own   conclusion was that **mitochondrial dysfunction is the shared MN   pathology across FUS and TARDBP**. Hits in lipid metabolism (PLA2   inhibitors, ingenol PKC modulators) are consistent.

**Categories that should be treated with extra skepticism:**

- **PMA, ionomycin, and other "common cell perturbants"**: these score   high in many L1000 reverse-signature analyses because they perturb   many genes non-specifically.
- **YM-155 (survivin inhibitor)**: scores highly across many   signature-reversal studies; signal may be cell-line-specific.
- Any **kinase-inhibitor "tool compound"** without a clinical phase.

## 7 · Limitations and what this does NOT do
- No motor-neuron LINCS data (L1000 platform uses cancer + immortalized lines).
- No validation in iPSC-MN rescue assays — purely computational ranking.
- Schweingruber DEGs are from isogenic iPSC at differentiation days 2-4;   post-mortem signatures would differ.
- Combination-therapy / dose-dependence not addressed.
- Statistical significance of L1000CDS² scores is not P-value-like;   treat as ranking only.

## 8 · Re-running
```bash
uv sync
uv run python scripts/01_load_signatures.py
uv run python scripts/02_lincs_query.py    # hits L1000CDS² (be polite, ~6 calls)
uv run python scripts/03_cross_validate.py
uv run python scripts/build_notebook.py
uv run jupyter lab notebooks/als_signature_reversal.ipynb
```